In [0]:
%sql

CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.BOSQL_MI_Mapping_V3 AS
WITH raw_data AS (
  SELECT DISTINCT
    ev3.universe,
    UPPER(TRIM(ev3.table)) AS BO_TABLE,
    CASE
      WHEN UPPER(ev3.SOURCE) LIKE 'ALIAS%' AND ev3.base_table IS NOT NULL AND ev3.base_table != ''
        THEN UPPER(TRIM(ev3.base_table))
      ELSE UPPER(TRIM(ev3.table))
    END AS source_table,
    UPPER(TRIM(ev3.schema)) AS SCHEMA,
    CASE
      WHEN UPPER(ev3.SOURCE) LIKE 'ALIAS%'
        THEN UPPER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(ev3.SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE UPPER(TRIM(ev3.SOURCE))
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction as ev3

  UNION ALL

  SELECT  DISTINCT
    ev4.universe,
    UPPER(TRIM(ev4.table)) AS BO_TABLE,
    CASE
      WHEN UPPER(ev4.SOURCE) LIKE 'ALIAS%' AND ev4.base_table IS NOT NULL AND ev4.base_table != ''
        THEN UPPER(TRIM(ev4.base_table))
      ELSE UPPER(TRIM(ev4.table))
    END AS source_table,
    UPPER(TRIM(ev4.schema)) AS SCHEMA,
    CASE
      WHEN UPPER(ev4.SOURCE) LIKE 'ALIAS%'
        THEN UPPER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(ev4.SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE UPPER(TRIM(ev4.SOURCE))
    END AS SOURCE

  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining as ev4
), 
ranked_source AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY universe, BO_TABLE, SOURCE_TABLE, SCHEMA
      ORDER BY
        CASE 
          WHEN SOURCE = 'SYMPHONY' THEN 1
          WHEN SOURCE = 'ORACLE FINANCE' THEN 2
          WHEN SOURCE = 'ORACLE FINANCE TABLE' THEN 3
          WHEN SOURCE = 'ORACLE FINANCE VIEW' THEN 4
          WHEN SOURCE = 'ORACLEFINANCE' THEN 5
          WHEN SOURCE = 'ORACLE FINANCE MATERIALIZED VIEW' THEN 6
          WHEN SOURCE = 'ORACLE FINACNE' THEN 7
          WHEN SOURCE = 'DW (FACT)' THEN 8
          WHEN SOURCE = 'DW' THEN 9
          WHEN SOURCE = 'DW (DIMENSION)' THEN 10
          WHEN SOURCE = 'DW (VIEW)' THEN 11
          WHEN SOURCE = 'INFORMATICA' THEN 12
          WHEN SOURCE = 'SYSTEM TABLE' THEN 13
          WHEN SOURCE = 'DERIVED' THEN 14
          WHEN SOURCE IN ('NA', 'NOT AVAILABLE') THEN 15
          ELSE 99
        END
    ) AS rn1
  FROM raw_data
),
deduped_source AS (
  SELECT universe, BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
  FROM ranked_source
  WHERE rn1 = 1
),
-- Level 2: Deduplicate by (BO_TABLE, SOURCE_TABLE) — keep highest priority SCHEMA
ranked_schema AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY BO_TABLE, SOURCE_TABLE
      ORDER BY
        CASE 
          WHEN SCHEMA = 'ORABUP0' THEN 1
          WHEN SCHEMA = 'ORADMART1' THEN 2
          WHEN SCHEMA = 'AR' THEN 3
          WHEN SCHEMA = 'GL' THEN 4
          WHEN SCHEMA = 'APPS' THEN 5
          WHEN SCHEMA = 'AP' THEN 6
          WHEN SCHEMA = 'CUST' THEN 7
          WHEN SCHEMA = 'FA' THEN 8
          WHEN SCHEMA = 'APPLSYS' THEN 9
          WHEN SCHEMA = 'PO' THEN 10
          WHEN SCHEMA = 'HR' THEN 11
          WHEN SCHEMA = 'XLA' THEN 12
          WHEN SCHEMA = 'JTF' THEN 13
          WHEN SCHEMA = 'INV' THEN 14
          WHEN SCHEMA = 'ZX' THEN 15
          WHEN SCHEMA = 'OKC' THEN 16
          WHEN SCHEMA = 'ICX' THEN 17
          WHEN SCHEMA = 'SYS' THEN 18
          WHEN SCHEMA = 'ORABOFP' THEN 19
          WHEN SCHEMA = 'GBRELS1' THEN 20
          WHEN SCHEMA IN ('NOT AVAILABLE', 'NOT AVILABLE') THEN 21
          ELSE 99
        END
    ) AS rn2
  FROM deduped_source
),
Universe_list as (
  SELECT universe, BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
  FROM ranked_schema
  WHERE rn2 = 1
  ORDER BY universe, BO_TABLE, SOURCE_TABLE ASC
)
select DISTINCT
  -- upper(trim(Universe_list.universe)) as UNIVERSE,
  upper(trim(Universe_list.BO_TABLE)) as BO_TABLE, 
  upper(trim(Universe_list.SOURCE_TABLE)) as MI_Table, 
  upper(trim(Universe_list.SCHEMA)) as MI_SCHEMA, 
  upper(trim(Universe_list.SOURCE)) as MI_SOURCE, 
  CASE
    WHEN upper(trim(Universe_list.SOURCE)) = 'SYMPHONY' 
      THEN upper(trim(Universe_list.SOURCE_TABLE))
    WHEN upper(trim(Universe_list.SOURCE)) LIKE '%ORACLE%' 
      THEN upper(trim(Universe_list.SOURCE_TABLE))
    ELSE COALESCE(upper(trim(d1.SOURCE_TABLE)), upper(trim(Universe_list.SOURCE_TABLE)))
  END as SOURCE_TABLE,
  CASE
    WHEN upper(trim(Universe_list.SOURCE)) = 'SYMPHONY' 
      THEN 'SYMPHONY'
    WHEN upper(trim(Universe_list.SOURCE)) LIKE '%ORACLE%' 
      THEN upper(trim(Universe_list.SCHEMA))
    ELSE COALESCE(upper(trim(d1.SOURCE_SCHEMA)), upper(trim(Universe_list.SCHEMA)))
  END as SCHEMA, 
  Universe_list.SOURCE as SOURCE
from Universe_list 
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO as D1
on upper(trim(Universe_list.SOURCE_TABLE)) = upper(trim(D1.TARGET_TABLE)) 
   and upper(trim(Universe_list.SCHEMA)) = upper(trim(D1.TARGET_SCHEMA))
order by upper(trim(BO_TABLE)) asc, upper(trim(SOURCE_TABLE)) asc


In [0]:
%sql
    
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO AS 
WITH RECURSIVE source_dict AS (
  SELECT DISTINCT
    upper(trim(TARGET_SCHEMA)) as TARGET_SCHEMA,
    upper(trim(TARGET_TABLE)) as TARGET_TABLE,
    upper(trim(SOURCE_SCHEMA)) as SOURCE_SCHEMA,
    upper(trim(SOURCE_TABLE)) as SOURCE_TABLE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.mi_dwh_data_dictionary
  WHERE SOURCE_SCHEMA IS NOT NULL 
    AND upper(trim(SOURCE_SCHEMA)) NOT IN ('CREATED AND DROPPED', 'SYS')
    AND TARGET_TABLE IS NOT NULL
    AND upper(trim(TARGET_SCHEMA)) NOT IN ('CREATED AND DROPPED')
),
lineage AS (
  -- Base case: all dictionary entries as starting points
  SELECT 
    TARGET_SCHEMA AS origin_target_schema,
    TARGET_TABLE AS origin_target_table,
    SOURCE_SCHEMA,
    SOURCE_TABLE,
    1 AS depth
  FROM source_dict

  UNION ALL

  -- Recursive case: trace SOURCE_TABLE through dictionary when it's in ORADMART1
  SELECT 
    l.origin_target_schema,
    l.origin_target_table,
    d.SOURCE_SCHEMA,
    d.SOURCE_TABLE,
    l.depth + 1
  FROM lineage l
  JOIN source_dict d
    ON l.SOURCE_TABLE = d.TARGET_TABLE 
    AND l.SOURCE_SCHEMA = d.TARGET_SCHEMA
  WHERE l.SOURCE_SCHEMA = 'ORADMART1'
    AND l.depth < 5  -- prevent infinite loops
)
SELECT DISTINCT
  origin_target_schema AS TARGET_SCHEMA,
  origin_target_table AS TARGET_TABLE,
  SOURCE_SCHEMA,
  SOURCE_TABLE
FROM lineage
WHERE SOURCE_SCHEMA != 'ORADMART1'  -- only keep real sources (not intermediate DWH tables)
ORDER BY TARGET_SCHEMA, TARGET_TABLE, SOURCE_SCHEMA, SOURCE_TABLE

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage as
select 
  upper(trim(v3.BO_TABLE)) as BO_SQL_TABLE, 
  upper(trim(v3.SOURCE_TABLE)) as MI_Table, 
  upper(trim(v3.SCHEMA)) as MI_SCHEMA, 
  upper(trim(v3.SOURCE)) as MI_SOURCE, 
  upper(trim(d1.SOURCE_TABLE)) as Src_table, 
  upper(trim(d1.SOURCE_SCHEMA)) as Src_schema 
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.BOSQL_MI_Mapping_V3 as V3
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO as D1
on upper(trim(v3.SOURCE_TABLE)) = upper(trim(D1.TARGET_TABLE)) 
   and upper(trim(v3.SCHEMA)) = upper(trim(D1.TARGET_SCHEMA))
order by upper(trim(v3.BO_TABLE)) asc, upper(trim(v3.SOURCE_TABLE)) asc
-- where D1.TARGET_TABLE is null

In [0]:
%sql 

create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.applications_databrickUC_Schema_mapping  AS
SELECT 'ORACLE FINANCE TABLE' AS BO_SOURCE, 'AP' AS BO_source_schema, 'bronze_raw_orf_ap' AS Databricks_Schema
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'APPLSYS', 'bronze_raw_orf_applsys'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'APPS', 'bronze_raw_orf_apps'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'AR', 'bronze_raw_orf_ar'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'CUST', 'bronze_raw_orf_cust'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'FA', 'bronze_raw_orf_fa'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'GL', 'bronze_raw_orf_gl'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'HR', 'bronze_raw_orf_hr'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'INV', 'bronze_raw_orf_inv'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'JTF', 'bronze_raw_orf_jtf'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'ORABOFP', 'bronze_raw_orf_orabofp'
UNION ALL
SELECT 'SYMPHONY', 'ORABUP0', 'bronze_raw_sym_orabup0'
UNION ALL
SELECT 'DW', 'ORABUP0', 'bronze_raw_symq_orabup0'
UNION ALL
SELECT 'DW', 'ORADMART1', 'bronze_raw_symq_oradmart1'
UNION ALL
SELECT 'DW', 'ORFM', NULL
UNION ALL
SELECT 'DW', 'ORFP', NULL
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'PO', 'bronze_raw_orf_po'
;

In [0]:
%sql
    
    
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage as
SELECT distinct
  upper(trim(aws.Document_Id)) as Document_Id,
  upper(trim(aws.Document_name)) as Document_name,
  wcd.cluster,
  upper(trim(aws.source_DB_connection)) as SAP_BO_Connection,
  upper(trim(aws.table_Name)) as BO_TABLE,
  upper(trim(aws.schema_Name)) as BO_SCHEMA,
  upper(trim(tl.MI_Table)) as MI_Table,
  upper(trim(tl.MI_SCHEMA)) as MI_SCHEMA,
  upper(trim(tl.MI_SOURCE)) as MI_SOURCE,
  upper(trim(tl.Src_table)) as Src_table,
  upper(trim(tl.Src_schema)) as Src_schema,
  COALESCE(upper(trim(tl.Src_table)), upper(trim(tl.MI_Table)), upper(trim(aws.table_Name))) as Calc_source_table,
  (CASE 
    WHEN upper(trim(tl.Src_table)) IS NULL AND upper(trim(tl.MI_Table)) IS NULL THEN upper(trim(aws.schema_Name))
    WHEN upper(trim(tl.Src_table)) IS NULL THEN upper(trim(tl.MI_SCHEMA))
    ELSE upper(trim(tl.Src_schema))
  END) as Calc_source_schema,
  upper(trim(aws.Document_CUID)) as Document_CUID,
  upper(trim(aws.Full_path)) as Full_path,
  upper(trim(aws.sql_table)) as sql_table,
  upper(trim(aws.updated_by)) as updated_by,
--   (case when upper(trim(DB_In.tableName)) is null then 'N' else 'Y' end) as Databrick_ingested,
  current_timestamp() AS linage_ingestion_ts
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source AS aws
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as wcd
  ON upper(trim(aws.Document_Id)) = upper(trim(wcd.Document_Id))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage AS tl
  ON upper(trim(aws.table_Name)) = upper(trim(tl.BO_SQL_TABLE))

In [0]:
%sql
    
    
ALTER TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage ADD COLUMN BO_DataConnection STRING;

MERGE INTO dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage AS awfl
USING (
  SELECT
    upper(trim(Document_Id)) AS Document_Id,
    concat_ws('|', array_sort(collect_set(upper(trim(Connection_Name))))) AS BO_DataConnection
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
  GROUP BY upper(trim(Document_Id))
) AS conn
ON upper(trim(awfl.Document_Id)) = conn.Document_Id
WHEN MATCHED THEN UPDATE SET awfl.BO_DataConnection = conn.BO_DataConnection;

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_UCflagged as 
SELECT distinct
 fl1.*, DB_schema.Databricks_Schema, (case when s1.table_schema is null or s1.table_name is null  then 'N' else 'Y' end) as databricks_ingested
from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage as fl1
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.applications_databrickUC_Schema_mapping as DB_schema
  ON 
    (
      upper(trim(fl1.Calc_source_schema)) = upper(trim(DB_schema.BO_source_schema))
      AND (
        upper(trim(fl1.Calc_source_schema)) <> 'ORABUP0'
        OR (
          upper(trim(fl1.Calc_source_schema)) = 'ORABUP0'
          AND upper(trim(split(fl1.MI_SOURCE, ' ')[0])) = upper(trim(DB_schema.BO_SOURCE))
        )
      )
    )
left join (SELECT distinct table_schema, table_name FROM dataplatform01_central_prd_catalog_01.information_schema.tables ) as s1
on upper(trim(DB_schema.Databricks_Schema))=upper(Trim(s1.table_schema)) and upper(trim(fl1.Calc_source_table))=upper(trim(s1.table_name))


